In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("benign_only.csv")
df.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
unique_items = df['Label'].unique()

print(df['Label'].value_counts())

Label
BENIGN    2273097
Name: count, dtype: int64


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

def prepare_cicids_data(csv_path, batch_size=256):
    df = pd.read_csv(csv_path)
    
    # Strip whitespace from column names (common issue in CICIDS2017)
    df.columns = df.columns.str.strip()
    
    # Drop categorical identifiers or labels if present
    cols_to_drop = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 
                    'Destination Port', 'Protocol', 'Timestamp', 'Label']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # Clean Inf and NaN values common in CICIDS2017
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(df.median(), inplace=True)
    
    # Standardize features (Mean=0, Std=1)
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df)
    
    # Convert to PyTorch Tensor DataLoader
    tensor_data = torch.tensor(scaled_data, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    return dataloader, scaler, tensor_data.shape[1]

In [ ]:
class PacketAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(PacketAutoencoder, self).__init__()
        
        # Encoder: Compresses packet feature representation
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 16) # Bottleneck layer
        )
        
        # Decoder: Reconstructs the packet feature vector
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, input_dim)
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [ ]:
def train_autoencoder(dataloader, input_dim, epochs=30, lr=1e-3):
    device = torch.device("xpu")
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PacketAutoencoder(input_dim).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in dataloader:
            inputs = batch[0].to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * inputs.size(0)
            
        epoch_loss = total_loss / len(dataloader.dataset)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.6f}")
            
    return model, device

In [ ]:
def set_threshold(model, dataloader, device, percentile=99.0):
    model.eval()
    errors = []
    criterion = nn.MSELoss(reduction='none') # Per-sample loss
    
    with torch.no_grad():
        for batch in dataloader:
            inputs = batch[0].to(device)
            reconstructed = model(inputs)
            # Calculate MSE loss per feature row
            mse = torch.mean((inputs - reconstructed) ** 2, dim=1)
            errors.extend(mse.cpu().numpy())
            
    errors = np.array(errors)
    threshold = np.percentile(errors, percentile)
    print(f"Calculated Anomaly Threshold ({percentile}th percentile): {threshold:.6f}")
    return threshold

def detect_anomalies(model, new_packet_csv, scaler, threshold, device):
    """
    Evaluates new unseen packet traffic and flags anomalies.
    """
    # Preprocess incoming dataset with the fitted scaler
    df_new = pd.read_csv(new_packet_csv)
    cols_to_drop = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 
                    'Destination Port', 'Protocol', 'Timestamp', 'Label']
    df_features = df_new.drop(columns=[col for col in cols_to_drop if col in df_new.columns])
    df_features.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_features.fillna(df_features.median(), inplace=True)
    
    scaled = scaler.transform(df_features)
    inputs = torch.tensor(scaled, dtype=torch.float32).to(device)
    
    model.eval()
    with torch.no_grad():
        reconstructed = model(inputs)
        mse_scores = torch.mean((inputs - reconstructed) ** 2, dim=1).cpu().numpy()
        
    predictions = (mse_scores > threshold).astype(int) # 1 = Anomaly, 0 = Normal
    return predictions, mse_scores

In [ ]:
# 1. Load normal dataset & prepare
dataloader, scaler, input_dim = prepare_cicids_data('benign_only.csv')

# 2. Train Autoencoder
model, device = train_autoencoder(dataloader, input_dim, epochs=10)

# 3. Derive threshold from normal training reconstruction loss
threshold = set_threshold(model, dataloader, device, percentile=99.0)

# 4. Predict on mixed test traffic CSV
# predictions, scores = detect_anomalies(model, 'test_packets.csv', scaler, threshold, device)